## Problem Overview

We are given a user behavior dataset where the task is to predict whether a user will convert (binary classification).

### Files:
- train.csv -> labeled training data
- public_test.csv -> validation-style test set (also has labels)
- private_test.csv -> final evaluation dataset (no labels)

### Goal:
Predict `Converted` (0/1) and maximize **F1-score**, which balances precision and recall.

## Step 1 - Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

## Step 2 - Load Data

I read all three CSV files — `train.csv`, `public_test.csv`, and `private_test.csv` — into pandas DataFrames.
The training set contains labelled examples we will learn from, while the private test set is what we need to make final predictions on.
Printing the shapes gives a quick sanity check that the files loaded correctly.


In [2]:
train = pd.read_csv("./dataset/train.csv")
public_test = pd.read_csv("./dataset/public_test.csv")
private_test = pd.read_csv("./dataset/private_test.csv")

## Step 3 - EDA

In [3]:
print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test.shape)

train.head()

Train Shape: (10000, 14)
Public Test Shape: (3000, 14)
Private Test Shape: (3000, 13)


,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


### Key Insights from Data Exploration

From initial analysis, I observed:

- Missing values exist in: Age, Income, Time_On_Site
- Users who convert show higher engagement:
  - More Pages_Viewed
  - More Products_Viewed
  - Higher Time_On_Site
- Strong categorical signals from Traffic_Source
- Moderate signal from Discount_Seen
- Pages_Viewed and Products_Viewed are highly correlated

### Key Conclusion:
Engagement features and missing-value patterns are strong predictors of conversion.

## Step 4 - Split Features and Target

Split training data into train/validation to:
- Prevent overfitting
- Tune threshold
- Evaluate model before public test

In [4]:
X = train.drop(columns=["User_ID", "Converted"])
y = train["Converted"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Step 5 - Preprocessing Pipeline

#### Data Preprocessing Strategy

I applied a consistent preprocessing pipeline to avoid data leakage:

- Missing value indicators
- Median imputation (train-only statistics)
- One-hot encoding for categorical variables
- Feature alignment between datasets

This ensures train, validation, and test data remain consistent in structure.

### 5.1 Copy datasets

In [5]:
X_train_clean = X_train.copy()
X_val_clean = X_val.copy()

During EDA I noticed missing values existed in:
- Age
- Income
- Time_On_Site

Handled them using:
1. Missing indicators
2. Median imputation (train-only statistics)

### 5.2 Missing Value Indicators

In [6]:
for col in ["Age", "Income", "Time_On_Site"]:
    X_train_clean[col + "_missing"] = X_train_clean[col].isnull().astype(int)
    X_val_clean[col + "_missing"] = X_val_clean[col].isnull().astype(int)

### 5.3 Median Imputation

In [7]:
impute_cols = ["Age", "Income", "Time_On_Site"]

medians = {col: X_train_clean[col].median() for col in impute_cols}

for col in impute_cols:
    X_train_clean[col] = X_train_clean[col].fillna(medians[col])
    X_val_clean[col] = X_val_clean[col].fillna(medians[col])

### 5.4 One-Hot Encoding

Found categorical variables:
- Device_Type
- Traffic_Source

Thus, used `One-Hot Encoding` for Logistic Regression compatibility.

In [8]:
cat_cols = ["Device_Type", "Traffic_Source"]

X_train_clean = pd.get_dummies(X_train_clean, columns=cat_cols, dtype=int)
X_val_clean = pd.get_dummies(X_val_clean, columns=cat_cols, dtype=int)

### 5.5 Align Train & Validation

In [9]:
X_train_clean, X_val_clean = X_train_clean.align(
    X_val_clean,
    join="left",
    axis=1,
    fill_value=0
)

## Step 6 - Feature Scaling
Appling `StandardScaler` because:
- Logistic Regression is sensitive to feature scale
- Helps convergence
- Stabilize gradient optimization
- Ensure fair weight learning across features

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_clean)
X_val_scaled = scaler.transform(X_val_clean)

## Step 7 - Model Training
Used __Logistic Regression__ with:
- class_weight = balanced (handles imbalance)
- increased max_iter (convergence stability)

In [11]:
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


## Step 8 - Threshold Tuning (F1 Optimization)

Instead of default threshold (0.5), I tuned classification threshold to maximize **F1-score**.

This is important because:
- Dataset is slightly imbalanced (~69%/31%)
- F1-score depends on precision-recall tradeoff
- Optimal threshold is data-dependent

In [12]:
probs = model.predict_proba(X_val_scaled)[:, 1]

best_f1 = 0
best_t = 0

for t in np.arange(0.1, 0.9, 0.01):
    preds = (probs >= t).astype(int)
    score = f1_score(y_val, preds)
    
    if score > best_f1:
        best_f1 = score
        best_t = t

print("Best F1:", best_f1)
print("Best Threshold:", best_t)

Best F1: 0.5835380835380836
Best Threshold: 0.46999999999999986


#### Final Validation Score

In [13]:
val_preds = (probs >= best_t).astype(int)
print("Validation F1:", f1_score(y_val, val_preds))

Validation F1: 0.5835380835380836


## Step 9 - Train Final Model (Full Data) for Submission

Trained final model on full dataset for submission.

In [14]:
X_full = X.copy()
y_full = y.copy()

### 9.1 Preprocessing (Full Data)

In [15]:
# Missing Value Indicators : 
for col in ["Age", "Income", "Time_On_Site"]:
    X_full[col + "_missing"] = X_full[col].isnull().astype(int)


# Median Imputation : 
medians = {col: X_full[col].median() for col in ["Age", "Income", "Time_On_Site"]}

for col in medians:
    X_full[col] = X_full[col].fillna(medians[col])


# One-Hot Encoding
X_full = pd.get_dummies(X_full, columns=cat_cols, dtype=int)

## Step 10 - Public Test Pipeline

In [16]:
X_public = public_test.drop(columns=["User_ID", "Converted"])


# missing indicators : 
for col in ["Age", "Income", "Time_On_Site"]:
    X_public[col + "_missing"] = X_public[col].isnull().astype(int)


# imputation : 
for col in medians:
    X_public[col] = X_public[col].fillna(medians[col])


# encoding
X_public = pd.get_dummies(X_public, columns=cat_cols, dtype=int)

In [17]:
# align : 
X_full, X_public = X_full.align(X_public, join="left", axis=1, fill_value=0)

In [18]:
# scaling :
scaler = StandardScaler()

X_full_scaled = scaler.fit_transform(X_full)
X_public_scaled = scaler.transform(X_public)

## Step 11 - Final Model Training

In [19]:
final_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_full_scaled, y_full)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


## Step 12 - Public Test Evaluation

In [20]:
public_probs = final_model.predict_proba(X_public_scaled)[:, 1]
public_preds = (public_probs >= best_t).astype(int)

print("Public Test F1:", f1_score(public_test["Converted"], public_preds))

Public Test F1: 0.5502781343602909


## Step 13 - Submission File

In [21]:
submission = pd.DataFrame({
    "User_ID": public_test["User_ID"],
    "Converted": public_preds
})

submission.to_csv("submission.csv", index=False)

## Final Insights

- Simple Logistic Regression performs strongly with proper preprocessing
- Missing value patterns carry predictive signal
- One-hot encoding + scaling significantly improves stability
- Threshold tuning provides major F1 improvement
- Feature engineering is less important than pipeline correctness in this dataset